## What MCP is ?

`MCP` (Model Context Protocol) is a standardized protocol for `connecting LLMs` (Large Language Models) or `AI agents` to `external tools`, `APIs`, and `data sources`. It defines how a client (like your RAG orchestrator or chatbot) and a server (hosting models, databases, or APIs) exchange messages. 
Think of it as the “plumbing” that makes sure your LLM can reliably call tools, fetch data, and stream results — with features like cancellation, retries, and progress updates.

#### Why MCP matters compared to REST/FastAPI ?

`REST/FastAPI` → You manually design endpoints (`/predict, /train, /delete`). Clients must know each endpoint and payload format. `MCP` → Provides a uniform protocol so any `LLM` client can discover and call server capabilities `without hard‑coding endpoints`.

#### When to use MCP?
If you’re building `LLM agents` that need to call multiple tools. If you want `standardized orchestration` across different models/services. If you need scalability and observability (streaming, retries, progress tracking).

- `FastAPI` → Great for serving ML models as REST endpoints.
- `MCP` → Great for connecting those endpoints into larger LLM/agent ecosystems with standardized communication.

## Multimodal agents or LLMs with tools

![Multimodal agents or LLMs with tools](image-5.png)

LLM’s : The core model — think of it as the “brain.” Image show the tools or data sources the LLM can call when it needs extra knowledge or functionality.
- `Arxiv`: Access to scientific papers for research.
- `Wiki`: General encyclopedic knowledge.
- `RAG DB`: A retrieval‑augmented generation database, where the LLM can fetch context‑specific documents.
- `DuckDuckGo` Search: A web search engine for real‑time information.

* Instead of being limited to what it was trained on, the LLM can **augment itself** by calling tools.
* This makes it more like an **agent** — able to reason, decide when to fetch external data, and combine multiple modalities (text, structured DBs, search).
* In practice, this is how modern AI systems handle **up-to-date knowledge, specialized domains, or multimodal tasks** (e.g., combining text with external structured data).

REST/FastAPI exposes `your model` as a tool. An `LLM agent` (like in the diagram) could call your model’s /predict endpoint when it needs a classification or forecast. At the same time, it could call other tools (search, RAG DB, wiki) to enrich its reasoning.

`For an example Tool Arxiv: Access to scientific papers for research. So LLM can read that research paper provided by tools Arxiv and provide answer to client according research paper provided by tools Arxiv.`

When an LLM is connected to a tool like **Arxiv**, it doesn't magically "know" all research papers. Instead:

* **The tool (Arxiv API)** provides access to papers — usually metadata, abstracts, or full text depending on permissions.
* **The LLM (agent)** can call that tool, retrieve the paper content, and then use its reasoning abilities to **summarize, explain, or answer questions** based on the retrieved text.
* **Client interaction:** When you (the client) ask a question like *"Explain the latest transformer architecture from Arxiv"*, the LLM agent can query Arxiv, fetch the relevant paper, and then generate an answer grounded in that paper.

![Workflow LLM Tools](image-6.png)

1. **Client asks a question** $\rightarrow$ "Summarize the new paper on diffusion models."
2. **LLM decides to use Arxiv tool** $\rightarrow$ Queries Arxiv API for that paper.
3. **Arxiv returns the paper content** $\rightarrow$ Abstract, sections, or full text.
4. **LLM processes the content** $\rightarrow$ Summarizes, highlights key points, answers your query.
5. **Response back to client** $\rightarrow$ "This paper proposes X, improves Y, and shows Z results."


The `LLM` doesn't store all research papers internally. Instead, **tools like Arxiv act as external knowledge sources**. The `LLM` uses them dynamically to fetch and reason over fresh information, then delivers an answer to the `client`. That’s exactly the idea behind **LLM + tools (agents)** — the model becomes more powerful by connecting to external resources like Arxiv, Wikipedia, RAG databases, or search engines.


## Basic about tool i.e. Arxiv

`Q. How Arxiv know which research paper need to access & provide to LLM. Is Arxiv use any algorithm or ML models. how the Arxiv know what contentment return form which research paper.`

### How Arxiv works

* Arxiv is a repository of scientific papers. It doesn't "decide" which paper you need; instead, it provides a **search API**.
* When a client (LLM, researcher, or app) sends a query — e.g., `"transformer architecture 2020"` — Arxiv's system searches its database of papers.
* It uses **information retrieval algorithms** (not ML models in most cases) to match your query against:
    * Titles
    * Abstracts
    * Author names
    * Categories (e.g., cs.CL for computational linguistics, cs.LG for machine learning)
The result is a list of papers that best match the query.

### Does Arxiv use ML?

* Traditionally, Arxiv’s search is based on **keyword matching and metadata indexing** (like a library catalog).
* More recently, advanced search engines may incorporate **ranking algorithms or semantic search** (using embeddings/ML models) to improve relevance.
* But the core idea is: Arxiv doesn’t “know” what you want — it just returns papers that match the query you (or the LLM) provide.


![message‑flow diagram MCP client and server interact with tools](image-7.png)

#### 1. MCP Host and Client (Top Box)

* This represents the **application or environment where the LLM runs** — for example:
    * **IDE integrations** like VS Code or Cursor.
    * **Desktop apps** like Claude Desktop.
    * **Custom applications** that embed an LLM.

* Inside this environment, the **MCP client** is responsible for sending and receiving messages using the MCP protocol.
* The **MCP host** manages the lifecycle — discovering available servers, handling authentication, and routing requests.
So, the **client** lives where the LLM or agent runs (your app, IDE, or desktop environment).

#### 2. MCP Server (Middle Box)

* The **MCP server** is the backend component that exposes tools, APIs, or databases to the LLM.
* It speaks the **MCP protocol** and uses a **codec** to encode/decode messages (for example, JSON or binary).
* The server can host multiple resources:
    * **Server 2** $\rightarrow$ **DB** (database access)
    * **Server 3** $\rightarrow$ **APIs** (external services like Arxiv, Wiki, RAG DB)
Each server provides a set of "tools" or "resources" that the LLM can call.

#### 3. Communication Flow

1. The **LLM (via MCP client)** sends a request — e.g., "Fetch latest Arxiv paper on diffusion models."
2. The **MCP host** routes this request through the **MCP protocol** to the correct server.
3. The **MCP server** executes the request (querying DBs or APIs) and returns the result.
4. The **client** receives the response and passes it back to the LLM for reasoning.


#### Where They Exist
* **MCP Client** $\rightarrow$ Lives with the LLM or app (frontend side: IDE, Claude Desktop, or your ML agent).
* **MCP Server** $\rightarrow$ Lives in the backend, exposing tools like Arxiv, Wiki, RAG DB, or APIs.
* **MCP Protocol** $\rightarrow$ The communication standard between them, ensuring consistency and reliability.


### Flow of Communication

1. **Client (LLM / App)**
    * The LLM or application receives a user query.
    * Example: “Get the latest Arxiv paper on diffusion models.”

2. **MCP Client (Protocol Layer)**
    * The client encodes the request using the **MCP protocol**.
    * It forwards the request to the MCP server.

3. **MCP Server (Routing Layer)**
    * The server receives the request.
    * It decides which tool to call (Arxiv, Wiki, RAG DB, or another API).

4. **Tool Execution (Arxiv, Wiki, DB, API)**
    * The chosen tool executes the query.
    * Example: Arxiv returns metadata and abstracts of relevant papers.

5. **Response Path Back**
    * The tool sends the response to the MCP server.
    * The MCP server packages the result and sends it back to the MCP client.
    * The MCP client delivers the final answer to the LLM/application.


## Build AI Assistant With MCP Servers And Tools Using LangChain And Groq